# MotoGP Grand Prix Race Winners - Data Preparation

This notebook cleans and standardizes the race winners dataset based on insights from the exploration phase.

## 0. Setup and Data Loading

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load raw data
raw_data_path = Path("../data/raw/grand_prix_race_winners.csv")
df_raw = pd.read_csv(raw_data_path)

print(f"Raw data loaded: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

## 1. Data Quality Assessment

In [ ]:
# Assess data quality
print("=== DATA QUALITY ASSESSMENT ===")
print(f"Total races: {len(df_raw)}")
print(f"Date range: {df_raw['Season'].min()} - {df_raw['Season'].max()}")
print(f"Unique circuits: {df_raw['Circuit'].nunique()}")
print(f"Unique riders: {df_raw['Rider'].nunique()}")
print(f"Unique constructors: {df_raw['Constructor'].nunique()}")
print(f"Unique countries: {df_raw['Country'].nunique()}")
print(f"Unique classes: {df_raw['Class'].nunique()}")

print(f"\nMissing values:")
print(df_raw.isnull().sum())

print(f"\nData types:")
print(df_raw.dtypes)

## 2. Column Standardization

In [ ]:
# Create working copy and standardize columns
df = df_raw.copy()

# Standardize column names
column_mapping = {
    'Circuit': 'circuit',
    'Class': 'class',
    'Constructor': 'constructor', 
    'Country': 'country',
    'Rider': 'rider',
    'Season': 'season'
}

df = df.rename(columns=column_mapping)

print("Standardized columns:")
print(list(df.columns))
df.head()

## 3. Data Type Corrections

In [ ]:
# Ensure proper data types
df['season'] = df['season'].astype(int)

# Clean text fields
text_columns = ['circuit', 'class', 'constructor', 'country', 'rider']
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

print("Updated data types:")
print(df.dtypes)

## 4. Circuit Name Standardization

In [ ]:
# Standardize circuit names
print("=== CIRCUIT NAME STANDARDIZATION ===")
print(f"Original unique circuits: {df['circuit'].nunique()}")

# Clean circuit names
df['circuit_clean'] = df['circuit'].str.strip()

# Show sample of circuit names
circuit_counts = df['circuit_clean'].value_counts()
print(f"\nTop 15 circuits by race count:")
print(circuit_counts.head(15))

# Check for potential duplicates or inconsistencies
similar_names = []
circuit_names = df['circuit_clean'].unique()

# Manual corrections for common inconsistencies
circuit_corrections = {
    # Add any specific corrections found during exploration
    # Example: 'Old Name': 'New Name'
}

if circuit_corrections:
    df['circuit_clean'] = df['circuit_clean'].replace(circuit_corrections)
    print(f"Applied {len(circuit_corrections)} circuit name corrections")

print(f"Final unique circuits: {df['circuit_clean'].nunique()}")

## 5. Constructor Name Standardization

In [ ]:
# Standardize constructor names
print("=== CONSTRUCTOR NAME STANDARDIZATION ===")
print(f"Original unique constructors: {df['constructor'].nunique()}")

df['constructor_clean'] = df['constructor'].str.strip()

# Show constructor distribution
constructor_counts = df['constructor_clean'].value_counts()
print(f"\nTop 15 constructors by wins:")
print(constructor_counts.head(15))

# Manual corrections for common inconsistencies
constructor_corrections = {
    # Add any specific corrections
    # Example: 'Yamaha Motor': 'Yamaha'
}

if constructor_corrections:
    df['constructor_clean'] = df['constructor_clean'].replace(constructor_corrections)
    print(f"Applied {len(constructor_corrections)} constructor name corrections")

print(f"Final unique constructors: {df['constructor_clean'].nunique()}")

## 6. Rider Name Standardization

In [ ]:
# Standardize rider names
print("=== RIDER NAME STANDARDIZATION ===")
print(f"Original unique riders: {df['rider'].nunique()}")

df['rider_clean'] = df['rider'].str.strip()

# Extract first and last names
df['rider_first_name'] = df['rider_clean'].str.split().str[0]
df['rider_last_name'] = df['rider_clean'].str.split().str[-1]
df['rider_display_name'] = df['rider_last_name'] + ', ' + df['rider_first_name']

# Show top riders
rider_counts = df['rider_clean'].value_counts()
print(f"\nTop 20 riders by wins:")
print(rider_counts.head(20))

# Check for potential duplicates
duplicate_riders = df['rider_clean'].duplicated()
if duplicate_riders.any():
    print(f"Note: Rider names are properly deduplicated by race occurrence")

print(f"Final unique riders: {df['rider_clean'].nunique()}")

## 7. Country and Continental Mapping

In [ ]:
# Standardize countries and add continental mapping
print("=== COUNTRY AND CONTINENTAL MAPPING ===")

df['country_clean'] = df['country'].str.strip().str.upper()

# Comprehensive continent mapping
continent_mapping = {
    # Europe
    'IT': 'Europe', 'ES': 'Europe', 'GB': 'Europe', 'FR': 'Europe', 'DE': 'Europe', 
    'NL': 'Europe', 'CH': 'Europe', 'AT': 'Europe', 'CZ': 'Europe', 'FI': 'Europe',
    'SE': 'Europe', 'NO': 'Europe', 'DK': 'Europe', 'BE': 'Europe', 'IE': 'Europe',
    'PT': 'Europe', 'GR': 'Europe', 'HU': 'Europe', 'PL': 'Europe', 'SI': 'Europe',
    'SK': 'Europe', 'HR': 'Europe', 'BG': 'Europe', 'RO': 'Europe', 'EE': 'Europe',
    'LV': 'Europe', 'LT': 'Europe', 'LU': 'Europe', 'MC': 'Europe', 'SM': 'Europe',
    
    # Asia
    'JP': 'Asia', 'MY': 'Asia', 'TH': 'Asia', 'CN': 'Asia', 'IN': 'Asia', 'ID': 'Asia',
    'KR': 'Asia', 'TW': 'Asia', 'PH': 'Asia', 'SG': 'Asia', 'VN': 'Asia', 'QA': 'Asia',
    'AE': 'Asia', 'SA': 'Asia', 'IL': 'Asia', 'TR': 'Asia', 'LB': 'Asia', 'IQ': 'Asia',
    'IR': 'Asia', 'AF': 'Asia', 'PK': 'Asia', 'BD': 'Asia', 'LK': 'Asia', 'MN': 'Asia',
    
    # Oceania
    'AU': 'Oceania', 'NZ': 'Oceania', 'PG': 'Oceania', 'FJ': 'Oceania',
    
    # North America
    'US': 'North America', 'CA': 'North America', 'MX': 'North America',
    'GT': 'North America', 'BZ': 'North America', 'CR': 'North America',
    'PA': 'North America', 'HN': 'North America', 'NI': 'North America',
    'SV': 'North America', 'JM': 'North America', 'CU': 'North America',
    
    # South America
    'BR': 'South America', 'AR': 'South America', 'VE': 'South America',
    'CO': 'South America', 'PE': 'South America', 'EC': 'South America',
    'UY': 'South America', 'PY': 'South America', 'BO': 'South America',
    'CL': 'South America', 'GY': 'South America', 'SR': 'South America',
    
    # Africa
    'ZA': 'Africa', 'EG': 'Africa', 'MA': 'Africa', 'TN': 'Africa',
    'DZ': 'Africa', 'LY': 'Africa', 'SD': 'Africa', 'KE': 'Africa',
    'GH': 'Africa', 'NG': 'Africa', 'ET': 'Africa', 'UG': 'Africa'
}

df['continent'] = df['country_clean'].map(continent_mapping)
df['continent'] = df['continent'].fillna('Other')

print(f"Country distribution:")
print(df['country_clean'].value_counts().head(15))

print(f"\nContinent distribution:")
print(df['continent'].value_counts())

# Check for unmapped countries
unmapped = df[df['continent'] == 'Other']['country_clean'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped countries: {unmapped}")
else:
    print("\nAll countries successfully mapped to continents")

## 8. Class Standardization

In [ ]:
# Standardize racing classes
print("=== CLASS STANDARDIZATION ===")

df['class_clean'] = df['class'].str.strip()

print(f"Class distribution:")
class_counts = df['class_clean'].value_counts()
print(class_counts)

# Create broader class categories
def categorize_class(class_name):
    if 'MotoGP' in class_name:
        return 'Premier Class'
    elif 'Moto2' in class_name:
        return 'Intermediate Class'
    elif 'Moto3' in class_name or '125' in class_name:
        return 'Lightweight Class'
    elif 'MotoE' in class_name:
        return 'Electric Class'
    elif '250' in class_name:
        return 'Historic Intermediate'
    elif '500' in class_name:
        return 'Historic Premier'
    else:
        return 'Other Class'

df['class_category'] = df['class_clean'].apply(categorize_class)

print(f"\nClass categories:")
print(df['class_category'].value_counts())

## 9. Temporal Analysis and Decade Mapping

In [ ]:
# Add temporal groupings
print("=== TEMPORAL ANALYSIS ===")

# Create decade grouping
df['decade'] = (df['season'] // 10) * 10

# Create era grouping
def categorize_era(season):
    if season < 1960:
        return '1950s Era'
    elif season < 1980:
        return 'Classic Era (1960-1979)'
    elif season < 2000:
        return 'Modern Era (1980-1999)'
    elif season < 2020:
        return 'Contemporary Era (2000-2019)'
    else:
        return 'Current Era (2020+)'

df['era'] = df['season'].apply(categorize_era)

print(f"Season range: {df['season'].min()} - {df['season'].max()}")
print(f"\nDecade distribution:")
print(df['decade'].value_counts().sort_index())

print(f"\nEra distribution:")
print(df['era'].value_counts())

## 10. Data Validation

In [ ]:
# Comprehensive data validation
print("=== DATA VALIDATION ===")

# Check season ranges
valid_seasons = (df['season'] >= 1949) & (df['season'] <= 2025)
invalid_seasons = (~valid_seasons).sum()
if invalid_seasons > 0:
    print(f"Warning: {invalid_seasons} races with invalid seasons")
else:
    print("✓ All seasons are within expected range")

# Check for completely missing data
required_fields = ['circuit_clean', 'class_clean', 'constructor_clean', 'rider_clean', 'season']
for field in required_fields:
    missing = df[field].isnull().sum()
    if missing > 0:
        print(f"Warning: {missing} missing values in {field}")
    else:
        print(f"✓ No missing values in {field}")

# Check for duplicates
duplicate_races = df.duplicated(subset=['season', 'circuit_clean', 'class_clean']).sum()
if duplicate_races > 0:
    print(f"Note: {duplicate_races} duplicate race combinations (possibly valid for multiple heats)")
else:
    print("✓ No duplicate race combinations")

print(f"\n=== VALIDATION SUMMARY ===")
print(f"Total races after preparation: {len(df)}")
print(f"Unique seasons: {df['season'].nunique()}")
print(f"Unique circuits: {df['circuit_clean'].nunique()}")
print(f"Unique riders: {df['rider_clean'].nunique()}")
print(f"Unique constructors: {df['constructor_clean'].nunique()}")
print(f"Unique classes: {df['class_clean'].nunique()}")

## 11. Final Dataset Structure

In [ ]:
# Define final column order
final_columns = [
    # Core data
    'season', 'circuit', 'circuit_clean', 'class', 'class_clean', 'class_category',
    'constructor', 'constructor_clean', 'rider', 'rider_clean', 
    'rider_first_name', 'rider_last_name', 'rider_display_name',
    'country', 'country_clean', 'continent',
    # Temporal groupings
    'decade', 'era'
]

# Create final dataset
df_final = df[final_columns].copy()

print("=== FINAL PREPARED DATASET ===")
print(f"Shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")

print(f"\nSample of prepared data:")
print(df_final.head(10))

print(f"\nData types:")
print(df_final.dtypes)

print(f"\nMissing values:")
print(df_final.isnull().sum())

## 12. Save Prepared Data

In [ ]:
# Create prepared data directory if it doesn't exist
prepared_data_path = Path("../../data/interim/")
prepared_data_path.mkdir(parents=True, exist_ok=True)

# Save prepared dataset
output_file = prepared_data_path / "race_winners_prepared.csv"
df_final.to_csv(output_file, index=False)

print(f"Prepared dataset saved to: {output_file}")
print(f"File size: {output_file.stat().st_size:,} bytes")

# Verification
df_verification = pd.read_csv(output_file)
print(f"\nVerification - loaded shape: {df_verification.shape}")
print("✓ File saved and verified successfully")

## 13. Preparation Summary

In [ ]:
print("=== PREPARATION SUMMARY ===")
print("\n✅ COMPLETED TASKS:")
print("1. Column name standardization")
print("2. Data type corrections and validation")
print("3. Circuit name cleaning and standardization")
print("4. Constructor name standardization")
print("5. Rider name cleaning with first/last name extraction")
print("6. Comprehensive country-continent mapping")
print("7. Racing class standardization and categorization")
print("8. Temporal groupings (decades, eras)")
print("9. Data validation and consistency checks")
print("10. Prepared dataset saved for analysis")

print("\n📊 DATASET IMPROVEMENTS:")
print(f"- Standardized names across all entities")
print(f"- Added continental mapping for geographical analysis")
print(f"- Created class categories for broader analysis")
print(f"- Added temporal groupings for era-based analysis")
print(f"- Extracted rider name components for flexible display")

print("\n🌍 GEOGRAPHICAL COVERAGE:")
continent_summary = df_final['continent'].value_counts()
for continent, count in continent_summary.items():
    percentage = (count / len(df_final)) * 100
    print(f"- {continent}: {count:,} races ({percentage:.1f}%)")

print("\n🏁 CLASS COVERAGE:")
class_summary = df_final['class_category'].value_counts()
for category, count in class_summary.items():
    percentage = (count / len(df_final)) * 100
    print(f"- {category}: {count:,} races ({percentage:.1f}%)")

print("\n➡️  READY FOR ANALYSIS PHASE")
print("The prepared race winners dataset is now ready for specific business question analysis.")